# NB02: Web of Microbes Strain Cross-Match

Cross-match 123 ENIGMA growth-curve strains against the Web of Microbes (WoM) exometabolomics database to identify strains with available metabolite production/consumption data.

In [1]:
from berdl_notebook_utils.setup_spark_session import get_spark_session
import pandas as pd, re

spark = get_spark_session()
print("Spark connected")

Spark connected


## 1. Load WoM organisms and growth-curve strains

In [2]:
# List all WoM organisms
wom_org = spark.sql("SELECT id, common_name, NCBI_taxid FROM kescience_webofmicrobes.organism ORDER BY id").toPandas()
print(f"WoM organisms: {len(wom_org)}")
wom_org

WoM organisms: 37


,id,common_name,NCBI_taxid
0,1,The Environment,NaN
1,2,Zymomonas mobilis strain ZM4 (ATCC 31821),NaN
2,3,Escherichia coli strain BW25113,NaN
3,4,"Theoretical: Escherichia coli JW0233 proA, pro...",NaN
4,5,"Theoretical: Escherichia coli JW2580 pheA, phe...",NaN
5,6,"Theoretical: Escherichia coli JW3745 ilvA, iso...",NaN
6,7,"Theoretical: Zymomonas mobilis ZMO0748 cysK, c...",NaN
7,8,Native microbiome: 018min Incubation,NaN
8,9,Native microbiome: 540min Incubation,NaN
9,10,Arthrobacter sp. (D1B45),NaN


In [3]:
# Load the 123 growth-curve strain names
strains_df = pd.read_csv(
    "/home/aparkin/BERIL-research-observatory/projects/genotype_to_phenotype_enigma/data/eda/per_strain_summary.tsv",
    sep="\t"
)
strain_names = sorted(strains_df["strain"].tolist())
print(f"Growth-curve strains: {len(strain_names)}")
print("Examples:", strain_names[:10])

Growth-curve strains: 123
Examples: ['EB106-03-03-XG65', 'EB106-05-01-XG146', 'EB106-05-01-XG201', 'EB106-07-01-XG149', 'EB271-G4-3-1', 'FW104-10B01', 'FW104-10F02', 'FW104-R3', 'FW215-L2', 'FW300-N1B4']


## 2. Cross-match: exact isolate-ID substring match

WoM organism names are formatted like `"Pseudomonas sp. (FW300-N2E3)"`. We extract the isolate ID from the WoM common_name and match against growth-curve strain names.

In [4]:
# Extract isolate IDs from WoM organism names
# Patterns: "Pseudomonas sp. (FW300-N2E3)", "Arthrobacter sp. (FW305-130)"
# Also check for direct matches (e.g. strain name appearing directly)

def extract_isolate_id(name):
    """Extract ENIGMA-style isolate ID from WoM organism name."""
    # Match parenthesized isolate IDs like (FW300-N2E3)
    m = re.search(r'\(([A-Z]+\d+[\w-]+)\)', name)
    if m:
        return m.group(1)
    return None

strain_set = set(strain_names)

matches = []
near_misses = []

for _, row in wom_org.iterrows():
    wom_name = row["common_name"]
    wom_id = row["id"]
    taxid = row["NCBI_taxid"]
    
    # Try extracting isolate ID
    isolate_id = extract_isolate_id(wom_name)
    
    if isolate_id and isolate_id in strain_set:
        matches.append({
            "strain": isolate_id,
            "wom_organism_id": wom_id,
            "wom_organism_name": wom_name,
            "ncbi_taxid": taxid,
            "match_type": "exact_isolate_id"
        })
    elif isolate_id:
        # Check for substring/prefix matches (near misses)
        for s in strain_names:
            # Same prefix family (e.g., FW300- vs FW300-)
            wom_prefix = re.match(r'([A-Z]+\d+)', isolate_id)
            strain_prefix = re.match(r'([A-Z]+\d+)', s)
            if wom_prefix and strain_prefix and wom_prefix.group(1) == strain_prefix.group(1):
                near_misses.append({
                    "growth_strain": s,
                    "wom_organism_name": wom_name,
                    "wom_isolate_id": isolate_id,
                })
    else:
        # Non-ENIGMA organism — check if strain name is a direct substring
        for s in strain_names:
            if s.lower() in wom_name.lower():
                matches.append({
                    "strain": s,
                    "wom_organism_id": wom_id,
                    "wom_organism_name": wom_name,
                    "ncbi_taxid": taxid,
                    "match_type": "direct_name"
                })

matches_df = pd.DataFrame(matches)
print(f"Exact matches: {len(matches_df)}")
if len(matches_df):
    print(matches_df[["strain", "wom_organism_name"]].to_string(index=False))

Exact matches: 6
     strain             wom_organism_name
 FW300-N2E3  Pseudomonas sp. (FW300-N2E3)
  GW456-L13   Pseudomonas sp. (GW456-L13)
 FW300-N2A2  Pseudomonas sp. (FW300-N2A2)
  GW456-L15   Pseudomonas sp. (GW456-L15)
FW507-14TSA Pseudomonas sp. (FW507-14TSA)
 FW300-N2F2  Pseudomonas sp. (FW300-N2F2)


In [5]:
# Show near-misses: same well-site prefix but different isolate
near_df = pd.DataFrame(near_misses).drop_duplicates()
if len(near_df):
    # Deduplicate — only show unique WoM organisms that didn't match
    wom_unmatched = near_df[["wom_organism_name", "wom_isolate_id"]].drop_duplicates()
    print(f"Near-misses (same site prefix, different isolate): {len(wom_unmatched)} WoM organisms")
    print(wom_unmatched.to_string(index=False))
else:
    print("No near-misses found")

Near-misses (same site prefix, different isolate): 3 WoM organisms
          wom_organism_name wom_isolate_id
Acidovorax sp. (GW101-3E06)     GW101-3E06
 Rhizobium sp. (GW101-3B10)     GW101-3B10
  Bacillus sp. (FW507-8R2A)     FW507-8R2A


## 3. Count metabolite observations per matched organism

In [6]:
if len(matches_df) > 0:
    matched_ids = matches_df["wom_organism_id"].tolist()
    id_list = ",".join(str(i) for i in matched_ids)
    
    # Count observations per organism
    obs_counts = spark.sql(f"""
        SELECT organism_id, 
               COUNT(*) as n_observations,
               COUNT(DISTINCT compound_id) as n_distinct_compounds,
               COUNT(DISTINCT environment_id) as n_environments,
               SUM(CASE WHEN action = 'I' THEN 1 ELSE 0 END) as n_increased,
               SUM(CASE WHEN action = 'E' THEN 1 ELSE 0 END) as n_emerged,
               SUM(CASE WHEN action = 'N' THEN 1 ELSE 0 END) as n_nochange
        FROM kescience_webofmicrobes.observation
        WHERE organism_id IN ({id_list})
        GROUP BY organism_id
    """).toPandas()
    
    # Merge with matches
    result = matches_df.merge(obs_counts, left_on="wom_organism_id", right_on="organism_id", how="left")
    result = result.drop(columns=["organism_id"])
    
    print(f"\n=== Summary ===")
    print(f"Matched strains: {len(result)} / 123")
    print(f"Total observations: {result['n_observations'].sum():,}")
    print(f"Total distinct compounds (across all): see per-strain below")
    print()
    print(result[["strain", "wom_organism_name", "n_observations", "n_distinct_compounds", 
                   "n_environments", "n_increased", "n_emerged", "n_nochange"]].to_string(index=False))
else:
    print("No matches found — nothing to count")
    result = matches_df


=== Summary ===
Matched strains: 6 / 123
Total observations: 630
Total distinct compounds (across all): see per-strain below

     strain             wom_organism_name  n_observations  n_distinct_compounds  n_environments  n_increased  n_emerged  n_nochange
 FW300-N2E3  Pseudomonas sp. (FW300-N2E3)             105                   105               1           31         27          47
  GW456-L13   Pseudomonas sp. (GW456-L13)             105                   105               1           49         34          22
 FW300-N2A2  Pseudomonas sp. (FW300-N2A2)             105                   105               1           26         32          47
  GW456-L15   Pseudomonas sp. (GW456-L15)             105                   105               1           44         25          36
FW507-14TSA Pseudomonas sp. (FW507-14TSA)             105                   105               1           44         33          28
 FW300-N2F2  Pseudomonas sp. (FW300-N2F2)             105                   105  

## 4. Identify distinct compounds measured for matched organisms

In [7]:
if len(matches_df) > 0:
    # Get overall compound stats for matched organisms
    compound_stats = spark.sql(f"""
        SELECT COUNT(DISTINCT o.compound_id) as total_distinct_compounds,
               COUNT(DISTINCT CASE WHEN c.compound_name NOT LIKE 'Unk_%' THEN o.compound_id END) as n_identified_compounds,
               COUNT(DISTINCT CASE WHEN c.compound_name LIKE 'Unk_%' THEN o.compound_id END) as n_unknown_compounds
        FROM kescience_webofmicrobes.observation o
        JOIN kescience_webofmicrobes.compound c ON o.compound_id = c.id
        WHERE o.organism_id IN ({id_list})
    """).toPandas()
    
    print("Compound coverage for matched organisms:")
    print(f"  Total distinct compounds: {compound_stats['total_distinct_compounds'].iloc[0]}")
    print(f"  Identified (named): {compound_stats['n_identified_compounds'].iloc[0]}")
    print(f"  Unknown (Unk_*): {compound_stats['n_unknown_compounds'].iloc[0]}")
    
    # Top 20 most common identified compounds across matched organisms
    top_compounds = spark.sql(f"""
        SELECT c.compound_name, COUNT(DISTINCT o.organism_id) as n_organisms, COUNT(*) as n_obs
        FROM kescience_webofmicrobes.observation o
        JOIN kescience_webofmicrobes.compound c ON o.compound_id = c.id
        WHERE o.organism_id IN ({id_list})
          AND c.compound_name NOT LIKE 'Unk_%'
          AND o.action IN ('I', 'E')
        GROUP BY c.compound_name
        ORDER BY n_organisms DESC, n_obs DESC
        LIMIT 20
    """).toPandas()
    
    print(f"\nTop 20 metabolites produced/increased (identified only):")
    print(top_compounds.to_string(index=False))

Compound coverage for matched organisms:
  Total distinct compounds: 105
  Identified (named): 105
  Unknown (Unk_*): 0



Top 20 metabolites produced/increased (identified only):
                       compound_name  n_organisms  n_obs
                      Pipecolic acid            6      6
                     N-acetyl-serine            6      6
                   2'-deoxyadenosine            6      6
                     trans-aconitate            6      6
                            arginine            6      6
                        nicotinamide            6      6
                              Malate            6      6
                             Guanine            6      6
                            thiamine            6      6
                             Adenine            6      6
              4-imidazoleacetic acid            6      6
                             glycine            6      6
                              Uracil            6      6
                            Cytosine            6      6
                             inosine            6      6
                           gua

## 5. Save results

In [8]:
outpath = "/home/aparkin/BERIL-research-observatory/projects/genotype_to_phenotype_enigma/data/eda/wom_strain_matches.tsv"

if len(result) > 0:
    result.to_csv(outpath, sep="\t", index=False)
    print(f"Saved {len(result)} matches to {outpath}")
else:
    # Save empty file with header
    pd.DataFrame(columns=["strain", "wom_organism_id", "wom_organism_name", "ncbi_taxid", 
                           "match_type", "n_observations", "n_distinct_compounds",
                           "n_environments", "n_increased", "n_emerged", "n_nochange"]).to_csv(outpath, sep="\t", index=False)
    print(f"No matches found. Saved empty TSV to {outpath}")

print("\nDone.")

Saved 6 matches to /home/aparkin/BERIL-research-observatory/projects/genotype_to_phenotype_enigma/data/eda/wom_strain_matches.tsv

Done.
